In [ ]:
# @title **Install dependencies (portable & non-fatal)**
# ---------------------------------------------------------------------------
# Installs into the CURRENT kernel via `sys.executable -m pip` so it targets the
# right Python in Colab AND in local venvs (a bare `!pip` can hit the wrong one).
# Wrapped in try/except so that if the machine is offline or a wheel fails, the
# notebook does NOT halt: the guarded imports in the next cell let it keep running
# against the cached dataset. `google-serp-api` is dropped (never imported);
# `python-dotenv` is added for local `.env` credential loading.
# ---------------------------------------------------------------------------
import sys, subprocess

PACKAGES = [
    "smolagents[toolkit]",   # multi-agent framework
    "google-search-results",  # provides `serpapi.GoogleSearch`
    "python-dotenv",          # local .env credential loading
]

try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PACKAGES],
        check=True, timeout=600,
    )
    print("[info] Dependencies installed into the active kernel.")
except Exception as exc:
    print(f"[info] Skipping install ({exc.__class__.__name__}); continuing with "
          "already-available packages (guarded imports handle any that are missing).")


**1. Setup and Agent Definition**


In [ ]:
# @title **LLM Setup (portable: Google Colab, local, or fully offline)**
# ---------------------------------------------------------------------------
# This cell was modified to run in three environments without edits:
#   1. Google Colab  -> reads secrets from google.colab.userdata
#   2. Local Jupyter -> reads secrets from environment vars / a local .env file
#   3. Offline/no-keys -> everything still runs against the cached dataset below
#
# CREDENTIAL SAFETY: no keys are ever printed. We only report set/missing.
# ---------------------------------------------------------------------------
import os
from typing import Dict, Any

# Master switch. Leave False to run reproducibly offline with cached data.
# Set True ONLY when you have live SerpAPI + LLM keys and want fresh searches.
RUN_LIVE = False


def _load_secret(name: str):
    """Load a credential from Colab userdata, then env vars / .env, else None."""
    # 1) Google Colab secrets manager
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # 2) Local: optionally hydrate os.environ from a .env file, then read it
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass
    return os.environ.get(name)


serp_api = _load_secret("serp_api")
GL_OpenAI = _load_secret("GL_OpenAI")

# smolagents / serpapi are OPTIONAL offline. Guard the imports so the whole
# notebook still executes top-to-bottom when they (or the keys) are missing.
try:
    from smolagents import OpenAIServerModel, CodeAgent, WebSearchTool, Tool, tool
    SMOLAGENTS_AVAILABLE = True
except Exception as exc:  # pragma: no cover - environment dependent
    print(f"[info] smolagents unavailable ({exc.__class__.__name__}); offline mode.")
    SMOLAGENTS_AVAILABLE = False

    # No-op stand-ins so the later @tool-decorated cells still define plain
    # Python functions we can call directly against the cached dataset.
    def tool(fn=None, **kwargs):
        return fn if callable(fn) else (lambda f: f)

    class _OfflineStub:
        def __init__(self, *a, **k):
            pass

        def run(self, *a, **k):
            raise RuntimeError("Live agents are unavailable offline; use cached results.")

    OpenAIServerModel = CodeAgent = WebSearchTool = Tool = _OfflineStub

try:
    from serpapi import GoogleSearch
except Exception:
    GoogleSearch = None  # only needed for live search tools

# Build a live LLM client only if we can AND want to.
model = None
if RUN_LIVE and SMOLAGENTS_AVAILABLE and GL_OpenAI:
    model = OpenAIServerModel(
        model_id="gpt-4o-mini",
        api_base="https://aibe.mygreatlearning.com/openai/v1",
        api_key=GL_OpenAI,
    )
    print("[info] Live model initialized.")
else:
    print(
        "[info] Offline mode -> "
        f"RUN_LIVE={RUN_LIVE}, smolagents={SMOLAGENTS_AVAILABLE}, "
        f"serp_api={'set' if serp_api else 'missing'}, "
        f"GL_OpenAI={'set' if GL_OpenAI else 'missing'}"
    )


## **Tools**

### **Define - Flight Tool**

In [ ]:
flight_system_prompt = """
You are a Travel Assistant Agent responsible for searching flight details between origin and destination locations for a given date and trip type.

For the flight search, you must use the `search_flights` tool with arguments:
- origin: airport code or city code where the flight departs from (e.g., 'CDG')
- destination: airport code or city code where the flight arrives (e.g., 'BER')
- date: departure date in YYYY-MM-DD format (e.g., '2025-10-10')
- trip_type: integer trip type (1=Round trip, 2=One way, 3=Multi-city)

Return the flight info formatted exactly as a dictionary with these keys:
{
    'Airline': 'easyJet',
    'Flight Number': 'U2 4631',
    'Departure Airport': 'Paris Charles de Gaulle Airport (CDG)',
    'Departure Time': 'October 10, 2025, at 07:25',
    'Arrival Airport': 'Berlin Brandenburg Airport (BER)',
    'Arrival Time': 'October 10, 2025, at 09:10',
    'Duration': '1 hour and 45 minutes',
    'Aircraft': 'Airbus A320',
    'Travel Class': 'Economy',
    'Legroom': '29 inches',
    'Price': '$122'
}

Do not add any additional text or commentary around the dictionary. Return only the required dictionary output formatted as shown above.
"""


In [ ]:
# @title --- Flight Agent ---
@tool
def search_flights(origin: str, destination: str, date: str, trip_type : int) -> str:
    """
    Fetches the flight details from google

    Args:
    origin (str): from where you want to take the flight
    destination (str): where you want to go
    date  (str): on which date want to take the flight
    trip_type (int): type of flight is it a one-way trip or a round trip, choose the options appropriately {"Round trip":1, "One way":2, "Multi-city":3}
    """

    params = {
            "api_key": serp_api,
            "engine": "google_flights",
            "hl": "en",
            "gl": "us",
            "departure_id": origin,
            "arrival_id": destination,
            "outbound_date": date,
            "type": trip_type,
            "currency": "USD"
    }

    results = GoogleSearch(params).get_dict()
    return results

flight_agent = CodeAgent(tools=[search_flights], model=model, name="flight_agent", description="Searches for flight details between two locations for a given date.", instructions=flight_system_prompt )

### **Define - Hotel Tool**

In [ ]:
hotel_system_prompt = """
You are a Travel Assistant Agent responsible for providing hotel information for nearby accommodations with key details.

Return a dictionary with the following keys and example values:
{
    "Hotel Name": "Hyatt Place Frankfurt Airport",
    "Star Rating": "5 Stars",
    "Price Per Night": "$95",
    "Distance from Airport": "15 min taxi",
    "Amenities": "Free Wi-Fi, Gym, Restaurant, Bar, Room service"
}

Do not add any additional text or commentary around the dictionary. Return only the required dictionary output formatted as shown above.
"""

In [ ]:
# @title --- Hotel Agent ---
@tool
def search_hotel(location: str, check_in_date: str, check_out_date: str) -> str:
    """
    Fetches the hotel details from google

    Args:
    location (str): Location where you want to search the hotels
    check_in_date (str): At what date do you want to check in
    check_out_date  (str): At what date do you want to check out
    """

    params = {
            "api_key": serp_api,
            "engine": "google_hotels",
            "q": location,
            "check_in_date": check_in_date,
            "check_out_date": check_out_date,
            "currency": "USD"
    }

    hotel_results = GoogleSearch(params).get_dict()
    return hotel_results

hotel_agent = CodeAgent(tools=[search_hotel], model=model, name="hotel_agent", description="Searches for hotels in the desired drop location of the flight", instructions=hotel_system_prompt )

## **Manger Agent**

In [ ]:
# @title --- Manager Agent orchestrates the workflow ---
manager_agent = CodeAgent(
    model=model,
    managed_agents=[flight_agent, hotel_agent],
    tools=[WebSearchTool()]
)

## **Agent Call**

### **Reproducible offline data**

The next cell defines `CACHED_RESULT`, a representative response used whenever `RUN_LIVE = False`. It lets every downstream cell run with no API keys and produce deterministic outputs. The values are shaped to demonstrate the assignment's central point — the cheapest *listed* option is not always the cheapest *total business cost* option.

In [ ]:
# @title --- Cached search results (used when RUN_LIVE is False) ---
# ---------------------------------------------------------------------------
# WHY THIS EXISTS
# The flight/hotel agents call live SerpAPI + an LLM, so results change every
# run and require paid keys. To make this notebook REPRODUCIBLE and runnable
# with no credentials, we cache one representative response here. All evaluation
# below runs against this dataset, so the graded numbers are deterministic.
#
# The data is deliberately shaped to expose the assignment's core lesson:
# the cheapest LISTED option is not the cheapest TOTAL-BUSINESS-COST option.
#   Flights: A is cheapest airfare but has 2 stops / 6h; C is priciest but nonstop.
#   Hotels:  X is cheapest/night but a long commute; Y is priciest but central.
#
# Fields are intentionally MESSY (prices as strings, durations in natural
# language, commute as mixed units) to exercise the robust parsers in Part 2.
# ---------------------------------------------------------------------------
CACHED_RESULT = {
    "flights": {
        "Option A": {
            "Airline": "BudgetWings", "Flight Number": "BW 88",
            "Departure Airport": "Paris Charles de Gaulle (CDG)",
            "Arrival Airport": "Berlin Brandenburg (BER)",
            "Duration": "6 hours", "Stops": 2, "Price": "$130",
        },
        "Option B": {
            "Airline": "EuroConnect", "Flight Number": "EC 210",
            "Departure Airport": "Paris Charles de Gaulle (CDG)",
            "Arrival Airport": "Berlin Brandenburg (BER)",
            "Duration": "3 hours and 30 minutes", "Stops": 1, "Price": "$210",
        },
        "Option C": {
            "Airline": "Lufthansa", "Flight Number": "LH 1032",
            "Departure Airport": "Paris Charles de Gaulle (CDG)",
            "Arrival Airport": "Berlin Brandenburg (BER)",
            "Duration": "1 hour and 50 minutes", "Stops": 0, "Price": "$340",
        },
    },
    "hotels": {
        "Hotels Found": [
            {
                "Hotel Name": "BER Airport Area North Inn", "Star Rating": "4.2 Stars",
                "Price Per Night": "$94", "Distance from Airport": "35 min taxi",
                "Commute Time": "90 minutes",  # long daily commute to the city center
                "Amenities": "Free Wi-Fi, Restaurant",
            },
            {
                "Hotel Name": "Berlin Mitte Comfort", "Star Rating": "4.0 Stars",
                "Price Per Night": "$140", "Distance from Airport": "20 min taxi",
                "Commute Time": "0.8 hours",  # moderate commute, mixed unit format
                "Amenities": "Free Wi-Fi, Gym, Bar",
            },
            {
                "Hotel Name": "City Center Grand", "Star Rating": "5 Stars",
                "Price Per Night": "$200", "Distance from Airport": "12 min taxi",
                "Commute Time": "18 min",  # central: short commute to business site
                "Amenities": "Spa, Restaurant, Bar, Gym",
            },
        ]
    },
}

print("[info] Cached dataset ready:",
      len(CACHED_RESULT["flights"]), "flights,",
      len(CACHED_RESULT["hotels"]["Hotels Found"]), "hotels.")


In [ ]:
# @title --- Example user query/task ---
task = """
Find me 3 flight options from Paris to Berlin, Germany that depart within the next 2 days,
covering a range from the cheapest available flight to the most expensive nonstop flight.
For each flight, provide airline, departure time, arrival time, duration, number of stops, and price.
Also, find 3 accommodation options for a 2-night stay in Berlin at varying distances from the
city-center business district, including hotel name, star rating, price per night, distance, and amenities.
"""

# Guarded execution: call the live manager agent only when we have keys AND opted in.
# Otherwise fall back to the cached dataset so the notebook is fully reproducible.
if RUN_LIVE and model is not None:
    result = manager_agent.run(task)
else:
    print("[info] RUN_LIVE is False -> using CACHED_RESULT (no API calls made).")
    result = CACHED_RESULT

result


In [ ]:
result

## **Goal Check**

1. Our objective is to verify whether the selected flights fall within the specified budget.

2. We need to confirm whether our goal is being met - is the agent accurately determining this?

In [ ]:
goal_check_system_prompt = """
You are a Goal Checker Agent tasked with evaluating flight options based on price.

Output should be a dictionary structured as follows:
{
  'Cheapest Flight': {'Price': float, 'Score': float, 'Status': str},
  'Most Expensive Flight': {'Price': float, 'Score': float, 'Status': str}
}

Your objectives:
1. Verify that each flight's price matches its budget status correctly ("Within Budget" or "Over Budget").
2. Assess the scores assigned to each flight option for validity within expected ranges.
3. Return a dictionary using the exact same structure as provided above, confirming the correctness or highlighting discrepancies.
4. Do not add any additional explanations, commentary, or reformattings outside the specified dictionary format.

Only return the final dictionary with keys 'Cheapest Flight' and 'Most Expensive Flight' and their respective nested dictionary values containing 'Price', 'Score', and 'Status'.
"""

In [ ]:
# @title --- Goal Based Agent - Flight Evaluator ---
@tool
def goal_checker(flight: Dict, budget_goal: float, penalty_factor: float) -> float:
    """
    Evaluate the flight based on budget goal

    Args:
    flight (dict): Contains the flight details like Price, Origin, Destination, Time, Date,...etc
    budget_goal (float): What is the budget, to be used for scoring the flight, the budget is in the USD
    penalty_factor (float): How to penalize the score if the flight is over budget
    """
    if isinstance(flight['Price'], str):
        price = float(flight['Price'].replace('$', ''))
    else:
        price = flight['Price']

    if price <= budget_goal:
        savings = budget_goal - price
        return {
            "Price":price,
            "Score": round(savings * (1.0 - penalty_factor), 2),
            "Savings": round(savings, 2)
            }

    else:
        excess = price - budget_goal
        return {
            "Price":price,
            "Score": round(-excess * penalty_factor, 2),
            "Difference": round(excess, 2)
        }

goal_checker_agent = CodeAgent(tools=[goal_checker], model=model, name="goal_checker_agent", description="Evaluate the flight based on budget goal", instructions=goal_check_system_prompt)

In [ ]:
# Original (price-only) goal-based flight evaluation.
budget_goal = 200.0     # USD airfare budget used by the original workflow
penalty_factor = 0.1    # penalty applied to over-budget airfare

evaluation_task = f"""
                  Your task is to check if the below provided flight details are within the budget or not
                  and return the penalty scores accordingly.
                  The budget for the flights is {budget_goal} and the penalty is {penalty_factor} for going above the budget.
                  Here are the Results : {result['flights']}
                  """

# Guarded: use the LLM agent live; offline, apply the ORIGINAL goal_checker tool
# directly to each flight so the price-only baseline is reproduced deterministically.
if RUN_LIVE and model is not None:
    goal_based_agent_response = goal_checker_agent.run(evaluation_task)
else:
    print("[info] Offline -> applying the original goal_checker tool to each flight.")
    goal_based_agent_response = {
        name: goal_checker(flight, budget_goal, penalty_factor)
        for name, flight in result["flights"].items()
    }

goal_based_agent_response


In [ ]:
goal_based_agent_response

## **Utility Check**

Here, we are evaluating the score based on two criteria:

1. The hotel's price (or price per night) must fall within our budget.

2. The hotel must have a 5-star rating.

In [ ]:
utility_checker_system_prompt ="""
You are a Utility-Based Agent tasked with evaluating a list of hotels based on their features and assigning a utility score to each hotel.

Input: A list of dictionaries representing hotels with keys such as 'Hotel Name', 'Star Rating', 'Price per Night', 'Distance from Airport', 'Amenities',..etc.

Your task:
1. Assess each hotel's attributes and calculate a utility score that reflects its overall value considering factors like rating, price, distance, and amenities.
2. Return a list of dictionaries where each dictionary contains only the keys:
   - 'Hotel' : the name of the hotel from 'Hotel Name'
   - 'utility_score' : the computed score for that hotel as an integer between 0 and 100, and always in the multiples of 50.

Output format example:
[
  {'Hotel': 'Hyatt Place Frankfurt Airport', 'utility_score': 100},
  {'Hotel': 'Steigenberger Airport Hotel Frankfurt', 'utility_score': 100},
  {'Hotel': 'Frankfurt Airport Marriott Hotel', 'utility_score': 0}
]

Important:
- Return only the output list formatted exactly as shown.
- Do not add any additional text or explanations.
"""

In [ ]:
# @title --- Utility Based Agent - Hotel Evaluator ---

from typing import Dict

@tool
def utility_checker(hotel: Dict, budget: float) -> Dict:
    """
    Evaluate the hotel based on star rating and budget to compute a utility score.

    Args:
        hotel (dict): Contains hotel details like 'Price Per Night', 'Star Rating', etc.
        budget (float): Budget to be used for scoring the hotel in USD.

    Returns:
        dict: Utility evaluation containing cumulative utility_score, parsed price, and scoring breakdown.
    """

    utility_score = 0

    # --- Star rating check ---
    stars_raw = hotel.get('Rating', 0) or hotel.get('Star Rating', 0)
    if ("5" in stars_raw) or (stars_raw == 5):
        utility_score += 50  # bonus for desired rating
    else:
        utility_score -= 50  # penalty for lower rating


    # --- Price check ---
    price_raw = hotel.get('Price Per Night', '0') or hotel.get('Price', '0') or hotel.get('Price per Night', '0')
    if isinstance(price_raw, str):
        price = float(price_raw.replace('$', '').replace(',', '').strip())
    else:
        price = float(price_raw)

    if price <= budget:
        utility_score += 50  # reward for meeting budget
    else:
        utility_score -= 50  # penalty for exceeding budget

    return utility_score


utility_checker_agent = CodeAgent(tools=[utility_checker], model=model, name="utility_checker_agent", description="Evaluate the hotels based on star rating and budget", instructions=utility_checker_system_prompt )

In [ ]:
# Original (price + star-rating only) utility-based hotel evaluation.
hotel_budget = 100.0   # USD per-night budget used by the original workflow

evaluation_task = f"""
                  Your task is to evaluate all the hotels individually, checking if the below provided hotel details
                  are within the budget or not and return the utility scores accordingly.
                  The budget for the hotel is ${hotel_budget}.
                  Here are the Results : {result['hotels']}
                  """

# Guarded: use the LLM agent live; offline, apply the ORIGINAL utility_checker tool
# directly to each hotel so the price+rating baseline is reproduced deterministically.
if RUN_LIVE and model is not None:
    utility_checker_agent_response = utility_checker_agent.run(evaluation_task)
else:
    print("[info] Offline -> applying the original utility_checker tool to each hotel.")
    utility_checker_agent_response = [
        {"Hotel": h.get("Hotel Name"), "utility_score": utility_checker(h, hotel_budget)}
        for h in result["hotels"]["Hotels Found"]
    ]

utility_checker_agent_response


In [ ]:
print(utility_checker_agent_response)

# **Part 2 — Labor-Aware Total Business Cost**

The original workflow (Part 1) ranks travel options by **listed price** only: the goal-based
agent scores flights against an airfare budget, and the utility-based agent scores hotels on
nightly price plus star rating. Neither one accounts for the **employee time** consumed by
travel and commuting — time the company pays for.

Part 2 extends the workflow to compute a **total business cost** for every option:

```
Total Flight Business Cost = Airfare + (Travel Hours x Employee Hourly Rate)
Total Hotel Business Cost  = Lodging Cost + (Commute Hours x Travel Days x Employee Hourly Rate)
```

The sections below add configurable labor parameters (Task 3), robust parsers for messy API
fields, business-cost functions for flights (Task 4) and hotels (Task 5), a labor-aware agent
prompt (Task 6), a before/after recommendation comparison (Task 7), and a sensitivity analysis
across several hourly rates (Task 8).

## **Task 3 — Employee Labor Cost Parameters**

All labor assumptions live in one place so the whole evaluation can be re-run by changing a
single value. Each choice is documented and justified in the code comments below.

In [ ]:
# @title --- Task 3: Configurable labor-cost parameters ---
# ---------------------------------------------------------------------------
# These parameters convert employee TIME into money. They are intentionally
# grouped in one cell so the entire analysis can be re-run by editing one value.
# The sensitivity analysis (Task 8) sweeps EMPLOYEE_HOURLY_RATE over a range.
# ---------------------------------------------------------------------------

# Base fully-loaded employee cost per hour (USD).
# Justification: a salaried professional earning ~$120-150k/yr, once benefits,
# payroll tax, and overhead are included (a "fully loaded" rate), costs the
# company roughly $70-80/hr against ~2,080 working hours/yr. $75 is a defensible
# mid-point and matches the value suggested in the assignment.
EMPLOYEE_HOURLY_RATE = 75.0

# Length of the hotel stay in nights (the task books a 2-night trip).
HOTEL_NIGHTS = 2

# Number of days the employee actually commutes to the business site.
# Justification: one commute cycle per on-site working day; for a 2-night trip
# we assume 2 commuting days. Kept separate from HOTEL_NIGHTS so an itinerary
# with a non-working travel day can be modeled without touching lodging math.
TRAVEL_DAYS = 2

# Fallback DAILY round-trip commute time (minutes) when a hotel record has no
# usable commute/distance field. Justification: 30 min is a conservative
# city-center commute; using an explicit default means missing data is handled
# transparently instead of being silently dropped (see estimate_commute_hours).
DEFAULT_COMMUTE_MINUTES = 30

# Lost-productivity penalty (hours) added PER stop, on top of the flight's
# elapsed duration. Justification: each connection adds boarding, security
# re-checks, and dead time that the elapsed "Duration" may understate. We use a
# modest 1.0 hr/stop rather than a larger figure to avoid over-penalizing
# connections; this is a documented assumption, not a measured value.
LAYOVER_PENALTY_HOURS = 1.0

# Assumption summary (also stated in the written response):
#  - Travel time counts as fully paid labor time.
#  - Layovers are captured via LAYOVER_PENALTY_HOURS x number of stops.
#  - Commute time is the hotel-to-business-district DAILY round trip.
#  - Missing/ambiguous data falls back to the documented defaults above.
print("Labor parameters set:")
print(f"  EMPLOYEE_HOURLY_RATE     = ${EMPLOYEE_HOURLY_RATE:.2f}/hr")
print(f"  HOTEL_NIGHTS             = {HOTEL_NIGHTS}")
print(f"  TRAVEL_DAYS              = {TRAVEL_DAYS}")
print(f"  DEFAULT_COMMUTE_MINUTES  = {DEFAULT_COMMUTE_MINUTES} min")
print(f"  LAYOVER_PENALTY_HOURS    = {LAYOVER_PENALTY_HOURS} hr/stop")


## **Robust parsers for messy travel data**

Travel APIs return inconsistent fields — prices as strings (`"$1,200"`), durations in natural
language (`"3 hours and 30 minutes"`), stops as text (`"nonstop"`), and sometimes missing
values. These helpers normalize that mess into numbers the cost functions can use. Missing data
is never silently dropped: it falls back to the documented defaults from Task 3 and is flagged.

In [ ]:
# @title --- Helper parsers (price, duration, stops, commute) ---
import re

def parse_price(value):
    """Convert a price such as '$1,250', '1250', or 1250.0 into a float (USD)."""
    if isinstance(value, (int, float)):
        return float(value)
    if value is None:
        raise ValueError("price is missing (None)")
    cleaned = re.sub(r"[^\d.]", "", str(value).replace(",", ""))
    if cleaned in ("", "."):
        raise ValueError(f"could not parse a price from {value!r}")
    return float(cleaned)

def parse_time_to_hours(text):
    """Parse a natural-language duration into hours; returns None if none found.
    Handles '6 hours', '3 hours and 30 minutes', '90 minutes', '0.8 hours',
    '18 min', '5h 30m', '2 hrs'."""
    if text is None:
        return None
    if isinstance(text, (int, float)):
        return float(text)
    s = str(text).lower()
    # capture every "<number> <unit>" pair and sum them in hours
    matches = re.findall(r"(\d+(?:\.\d+)?)\s*(hours?|hrs?|hr|h|minutes?|mins?|min|m)\b", s)
    if not matches:
        return None
    hours = 0.0
    for num, unit in matches:
        num = float(num)
        hours += num if unit.startswith("h") else num / 60.0
    return hours

def parse_stops(value):
    """Return number of stops as int. 'nonstop'/'direct' -> 0; '2 stops' -> 2."""
    if isinstance(value, (int, float)):
        return int(value)
    if value is None:
        return 0
    s = str(value).lower()
    if any(w in s for w in ("nonstop", "non-stop", "direct")):
        return 0
    m = re.search(r"\d+", s)
    return int(m.group()) if m else 0

def estimate_travel_hours(flight, layover_penalty_hours=LAYOVER_PENALTY_HOURS):
    """Total EFFECTIVE travel hours = elapsed duration + (stops x layover penalty).
    Falls back to a documented 3.0h assumption if Duration is unusable."""
    base = parse_time_to_hours(flight.get("Duration"))
    assumed = base is None
    if assumed:
        base = 3.0  # documented fallback when duration is missing/unparseable
    stops = parse_stops(flight.get("Stops", flight.get("stops", 0)))
    effective = base + stops * layover_penalty_hours
    return {
        "base_hours": round(base, 4),
        "stops": stops,
        "penalty_hours": round(stops * layover_penalty_hours, 4),
        "travel_hours": round(effective, 4),
        "duration_assumed": assumed,
    }

def estimate_commute_hours(hotel, default_commute_minutes=DEFAULT_COMMUTE_MINUTES):
    """Estimate DAILY round-trip commute hours from the hotel record.
    Prefers an explicit 'Commute Time'; otherwise uses the documented default and
    flags it (missing data is handled transparently, never silently ignored)."""
    raw = hotel.get("Commute Time") or hotel.get("Commute") or hotel.get("commute_time")
    hours = parse_time_to_hours(raw)
    if hours is None:
        return {"commute_hours": round(default_commute_minutes / 60.0, 4),
                "commute_assumed": True}
    return {"commute_hours": round(hours, 4), "commute_assumed": False}

print("Parsers defined: parse_price, parse_time_to_hours, parse_stops, "
      "estimate_travel_hours, estimate_commute_hours")


In [ ]:
# @title --- Robustness check: messy + missing inputs ---
# Demonstrates that the parsers handle the field formats the assignment calls out,
# and that MISSING data falls back to documented defaults (never silently dropped).

print("Prices:")
for v in ["$1,200", "$94", "1250", 340]:
    print(f"  {v!r:10} -> {parse_price(v):.2f}")
print("  'N/A'      -> handled:", end=" ")
try:
    parse_price("N/A")
except ValueError as e:
    print(f"ValueError ({e})")

print("\nDurations -> hours:")
for v in ["6 hours", "3 hours and 30 minutes", "1 hour and 50 minutes", "90 minutes", "0.8 hours", "18 min"]:
    print(f"  {v!r:26} -> {parse_time_to_hours(v)}")

print("\nFlight effective travel hours (duration + stops x penalty):")
for f in [{"Duration": "5 hours and 30 minutes", "Stops": "1 stop"},
          {"Duration": None, "Stops": 0}]:   # 2nd row: missing duration -> assumed
    info = estimate_travel_hours(f)
    tag = " (duration ASSUMED)" if info["duration_assumed"] else ""
    print(f"  {str(f):45} -> {info['travel_hours']} h{tag}")

print("\nHotel commute hours:")
for h in [{"Commute Time": "90 minutes"}, {"Hotel Name": "No-commute-field Inn"}]:
    info = estimate_commute_hours(h)
    tag = " (DEFAULT used)" if info["commute_assumed"] else ""
    print(f"  {str(h):45} -> {info['commute_hours']} h{tag}")


## **Task 4 — Labor-aware flight evaluation**

For each flight we now compute airfare, effective travel hours, the labor cost of that time, and
the **total flight business cost = airfare + travel hours x hourly rate**. The table makes it
explicit whether the lowest-airfare flight is also the lowest-total-cost flight.

In [ ]:
# @title --- Task 4: calculate_flight_business_cost ---
import pandas as pd

def calculate_flight_business_cost(flight, employee_hourly_rate=EMPLOYEE_HOURLY_RATE):
    """Total flight business cost = airfare + labor cost of travel time.

    labor cost = effective travel hours x hourly rate, where effective travel
    hours = elapsed duration + (number of stops x LAYOVER_PENALTY_HOURS).
    Returns a full breakdown so the reasoning is auditable."""
    airfare = parse_price(flight.get("Price"))                 # $ -> float
    t = estimate_travel_hours(flight)                          # duration + layover penalty
    labor = t["travel_hours"] * employee_hourly_rate           # paid time during travel
    total = airfare + labor
    return {
        "Airfare": round(airfare, 2),
        "Base Hours": t["base_hours"],
        "Stops": t["stops"],
        "Travel Hours": t["travel_hours"],     # effective (incl. layover penalty)
        "Labor Cost": round(labor, 2),
        "Total Business Cost": round(total, 2),
        "Duration Assumed": t["duration_assumed"],
    }

def evaluate_flights(flights, rate):
    """Return (DataFrame, cheapest-airfare key, lowest-total key)."""
    rows = {name: calculate_flight_business_cost(f, rate) for name, f in flights.items()}
    df = pd.DataFrame(rows).T
    cheapest_airfare = df["Airfare"].astype(float).idxmin()
    lowest_total = df["Total Business Cost"].astype(float).idxmin()
    return df, cheapest_airfare, lowest_total

flight_df, cheapest_airfare, lowest_total_flight = evaluate_flights(
    result["flights"], EMPLOYEE_HOURLY_RATE)

print(f"At ${EMPLOYEE_HOURLY_RATE:.0f}/hr:")
print(f"  Lowest airfare        : {cheapest_airfare} (${flight_df.loc[cheapest_airfare,'Airfare']:.0f})")
print(f"  Lowest TOTAL business : {lowest_total_flight} (${flight_df.loc[lowest_total_flight,'Total Business Cost']:.0f})")
if cheapest_airfare != lowest_total_flight:
    print(f"  -> The cheapest airfare is NOT the cheapest total: labor cost changes the winner.")
else:
    print(f"  -> The cheapest airfare is also the cheapest total business cost.")
flight_df


## **Task 5 — Labor-aware hotel evaluation**

For each hotel we compute lodging cost, daily commute labor, and the **total hotel business
cost = lodging + commute hours x travel days x hourly rate**. As with flights, the table shows
whether the cheapest nightly rate is still the best once commute labor is included.

In [ ]:
# @title --- Task 5: calculate_hotel_business_cost ---

def calculate_hotel_business_cost(hotel, employee_hourly_rate=EMPLOYEE_HOURLY_RATE,
                                  hotel_nights=HOTEL_NIGHTS, travel_days=TRAVEL_DAYS):
    """Total hotel business cost = lodging cost + commute labor cost.

    lodging cost      = price per night x nights
    commute labor cost = daily commute hours x travel days x hourly rate
    Returns a full breakdown so the reasoning is auditable."""
    ppn = parse_price(hotel.get("Price Per Night"))            # $ -> float
    lodging = ppn * hotel_nights
    c = estimate_commute_hours(hotel)                          # daily round-trip hours
    commute_labor = c["commute_hours"] * travel_days * employee_hourly_rate
    total = lodging + commute_labor
    return {
        "Price/Night": round(ppn, 2),
        "Nights": hotel_nights,
        "Lodging Cost": round(lodging, 2),
        "Commute Hrs/Day": c["commute_hours"],
        "Commute Labor": round(commute_labor, 2),
        "Total Business Cost": round(total, 2),
        "Commute Assumed": c["commute_assumed"],
    }

def evaluate_hotels(hotels, rate, nights=HOTEL_NIGHTS, days=TRAVEL_DAYS):
    """Return (DataFrame, cheapest-per-night name, lowest-total name)."""
    rows = {h["Hotel Name"]: calculate_hotel_business_cost(h, rate, nights, days)
            for h in hotels}
    df = pd.DataFrame(rows).T
    cheapest_night = df["Price/Night"].astype(float).idxmin()
    lowest_total = df["Total Business Cost"].astype(float).idxmin()
    return df, cheapest_night, lowest_total

hotel_df, cheapest_night, lowest_total_hotel = evaluate_hotels(
    result["hotels"]["Hotels Found"], EMPLOYEE_HOURLY_RATE)

print(f"At ${EMPLOYEE_HOURLY_RATE:.0f}/hr:")
print(f"  Cheapest per night    : {cheapest_night} (${hotel_df.loc[cheapest_night,'Price/Night']:.0f}/nt)")
print(f"  Lowest TOTAL business : {lowest_total_hotel} (${hotel_df.loc[lowest_total_hotel,'Total Business Cost']:.0f})")
if cheapest_night != lowest_total_hotel:
    print(f"  -> The cheapest nightly rate is NOT the cheapest total: commute labor changes the winner.")
else:
    print(f"  -> The cheapest nightly rate is also the cheapest total business cost.")
hotel_df
